In [ ]:
!pip install datasets numpy

# Imports

In [ ]:
import os
import re
import pickle
import json
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Optional
import numpy as np
from datasets import load_dataset

# Cellule 3 : Classe MalagasyNGramPredictor

In [ ]:
class MalagasyNGramPredictor:
    """
    Prédicteur du mot suivant basé sur des N-grammes
    Supporte les bigrammes et trigrammes avec lissage de Laplace
    """

    def __init__(self, n: int = 3, smoothing: float = 1.0):
        self.n = n
        self.smoothing = smoothing
        self.vocab = set()
        self.vocab_size = 0
        self.unigrams = Counter()
        self.ngrams = defaultdict(Counter)

        # Tokens spéciaux
        self.START = "<S>"
        self.END = "</S>"
        self.UNK = "<UNK>"

    def tokenize(self, text: str) -> List[str]:
        """Tokenisation simple pour le malgache"""
        text = text.lower()
        tokens = re.findall(r"[a-zA-Zàâäéèêëïîôöùûüçñ']+", text)
        tokens = [t for t in tokens if len(t) > 1 or t in ['a', 'i', 'ny']]
        return tokens

    def prepare_ngrams(self, tokens: List[str]) -> List[Tuple]:
        padded = [self.START] * (self.n - 1) + tokens + [self.END]
        ngrams = []
        for i in range(len(padded) - self.n + 1):
            context = tuple(padded[i:i + self.n - 1])
            word = padded[i + self.n - 1]
            ngrams.append((context, word))
        return ngrams

    def train(self, sentences: List[str], vocab_min_freq: int = 2):
        print(f" Entraînement du modèle {self.n}-grammes...")

        # Comptage des unigrammes
        for sent in sentences:
            tokens = self.tokenize(sent)
            self.unigrams.update(tokens)

        # Construction du vocabulaire
        self.vocab = {word for word, freq in self.unigrams.items()
                      if freq >= vocab_min_freq}
        self.vocab.add(self.START)
        self.vocab.add(self.END)
        self.vocab.add(self.UNK)
        self.vocab_size = len(self.vocab)

        print(f" Vocabulaire: {self.vocab_size} mots uniques")

        # Comptage des N-grammes
        total_ngrams = 0
        for sent in sentences:
            tokens = self.tokenize(sent)
            tokens = [t if t in self.vocab else self.UNK for t in tokens]
            ngrams = self.prepare_ngrams(tokens)

            for context, word in ngrams:
                self.ngrams[context][word] += 1
                total_ngrams += 1

        print(f" {total_ngrams} N-grammes enregistrés")
        print(f" {len(self.ngrams)} contextes uniques")

    def predict(self, context_tokens: List[str], top_k: int = 5) -> List[Tuple[str, float]]:
        if len(context_tokens) > self.n - 1:
            context_tokens = context_tokens[-(self.n - 1):]
        elif len(context_tokens) < self.n - 1:
            context_tokens = [self.START] * (self.n - 1 - len(context_tokens)) + context_tokens

        context_tokens = [t if t in self.vocab else self.UNK for t in context_tokens]
        context = tuple(context_tokens)

        probas = {}
        for word in self.vocab:
            count_w = self.ngrams[context].get(word, 0)
            total_context = sum(self.ngrams[context].values())
            prob = (count_w + self.smoothing) / (total_context + self.smoothing * self.vocab_size)
            probas[word] = prob

        probas = [(w, p) for w, p in sorted(probas.items(), key=lambda x: -x[1])
                  if w not in [self.START, self.END, self.UNK]]

        return probas[:top_k]

    def autocomplete(self, text: str, top_k: int = 5) -> List[Tuple[str, float]]:
        tokens = self.tokenize(text)
        if not tokens:
            return []
        return self.predict(tokens, top_k)

    def get_perplexity(self, sentences: List[str]) -> float:
        log_prob_sum = 0
        total_words = 0

        for sent in sentences:
            tokens = self.tokenize(sent)
            tokens = [t if t in self.vocab else self.UNK for t in tokens]
            ngrams = self.prepare_ngrams(tokens)

            sent_log_prob = 0
            for context, word in ngrams:
                count_w = self.ngrams[context].get(word, 0)
                total_context = sum(self.ngrams[context].values())
                prob = (count_w + self.smoothing) / (total_context + self.smoothing * self.vocab_size)
                sent_log_prob += np.log(prob)

            log_prob_sum += sent_log_prob
            total_words += len(tokens)

        perplexity = np.exp(-log_prob_sum / total_words)
        return perplexity

    def save(self, path: str):
        model_data = {
            'n': self.n,
            'smoothing': self.smoothing,
            'vocab': list(self.vocab),
            'vocab_size': self.vocab_size,
            'unigrams': dict(self.unigrams),
            'ngrams': {str(k): dict(v) for k, v in self.ngrams.items()},
            'START': self.START,
            'END': self.END,
            'UNK': self.UNK
        }
        with open(path, 'wb') as f:
            pickle.dump(model_data, f)
        print(f" Modèle sauvegardé dans {path}")

    def load(self, path: str):
        with open(path, 'rb') as f:
            model_data = pickle.load(f)

        self.n = model_data['n']
        self.smoothing = model_data['smoothing']
        self.vocab = set(model_data['vocab'])
        self.vocab_size = model_data['vocab_size']
        self.unigrams = Counter(model_data['unigrams'])
        self.ngrams = defaultdict(Counter)

        for k, v in model_data['ngrams'].items():
            context = eval(k)
            self.ngrams[context] = Counter(v)

        self.START = model_data['START']
        self.END = model_data['END']
        self.UNK = model_data['UNK']

        print(f" Modèle chargé depuis {path}")
        print(f"   {self.n}-grammes, vocabulaire: {self.vocab_size} mots")

# Cellule 4 : Classe MarkovChainPredictor (optionnelle)

In [ ]:
class MarkovChainPredictor(MalagasyNGramPredictor):
    """
    Version simplifiée basée sur une chaîne de Markov (bigrammes)
    """

    def __init__(self):
        super().__init__(n=2)
        self.transition_matrix = defaultdict(Counter)

    def train(self, sentences: List[str], vocab_min_freq: int = 2):
        super().train(sentences, vocab_min_freq)

        for context, word in self.ngrams.items():
            total = sum(self.ngrams[context].values())
            for w, count in self.ngrams[context].items():
                self.transition_matrix[context[0]][w] = count / total

    def predict_next(self, last_word: str, top_k: int = 5) -> List[Tuple[str, float]]:
        last_word = last_word if last_word in self.vocab else self.UNK
        if last_word not in self.transition_matrix:
            return []
        probas = self.transition_matrix[last_word].most_common(top_k)
        return [(w, p) for w, p in probas if w not in [self.START, self.END, self.UNK]]

# Cellule 5 : Fonction de chargement du dataset

In [ ]:
def load_malagasy_dataset():
    """Charge le dataset depuis Hugging Face"""
    print(" Chargement du dataset...")

    dataset = load_dataset(path='Lo-Renz-O/malagasy-sentence')

    sentences = [item['text'] for item in dataset['train']]
    print(f" {len(sentences)} phrases chargées")

    # Aperçu
    print("\n Exemples de phrases:")
    for i, sent in enumerate(sentences[:5]):
        print(f"   {i+1}. {sent[:100]}...")

    return sentences

# Cellule 6 : Fonction de division train/test

In [ ]:
def split_data(sentences: List[str], train_ratio: float = 0.9):
    split_idx = int(len(sentences) * train_ratio)
    train = sentences[:split_idx]
    test = sentences[split_idx:]
    print(f"\n Division des données:")
    print(f"   Train: {len(train)} phrases")
    print(f"   Test: {len(test)} phrases")
    return train, test

# Cellule 7 : Fonction de démonstration

In [ ]:
def demo(model, test_sentences=None):
    print("\n" + "="*60)
    print(" DÉMONSTRATION - Autocomplétion Malagasy")
    print("="*60)

    # Exemples de débuts de phrases
    examples = [
        "Ny", "Mbola", "Amin", "Tsy", "Raha",
        "Fa", "Eo", "Ao", "Ho", "Tonga"
    ]

    print("\n Prédictions à partir de débuts de phrases:\n")
    for ex in examples:
        preds = model.autocomplete(ex, top_k=5)
        print(f" '{ex}' → ", end="")
        if preds:
            suggestions = [f"{w} ({p:.3f})" for w, p in preds[:3]]
            print(", ".join(suggestions))
        else:
            print("(aucune prédiction)")

    # Contexte plus long
    print("\n Prédiction en contexte:")
    contexts = ["Ny tenin", "Aza manao", "Misy olona", "Tonga ny"]
    for ctx in contexts:
        preds = model.autocomplete(ctx, top_k=3)
        print(f"   '{ctx}' → {[w for w,_ in preds]}")

    # Mode interactif (optionnel)
    print("\n Mode interactif - Tapez du texte pour des suggestions (tapez 'quit' pour quitter)")
    while True:
        user_input = input("  Votre texte: ").strip()
        if user_input.lower() == 'quit':
            break
        preds = model.autocomplete(user_input, top_k=5)
        if preds:
            print(" Suggestions:", "  ".join([f"'{w}' ({p:.2%})" for w, p in preds[:5]]))
        else:
            print("  Aucune suggestion")

# Cellule 8 : Fonction principale et exécution

In [ ]:
def main():
    print("="*60)
    print(" MODÈLE D'AUTOCOMPLÉTION POUR LE MALGACHE")
    print("   N-grammes avec lissage de Laplace")
    print("="*60)

    # 1. Chargement des données
    sentences = load_malagasy_dataset()

    # 2. Split train/test
    train, test = split_data(sentences, train_ratio=0.9)

    # 3. Entraînement du modèle trigrammes
    print("\n" + "-"*40)
    print(" Modèle Trigrammes (3-grammes)")
    print("-"*40)

    trigram_model = MalagasyNGramPredictor(n=3, smoothing=0.5)
    trigram_model.train(train, vocab_min_freq=2)

    # 4. Évaluation sur un sous-ensemble du test
    perplexity = trigram_model.get_perplexity(test[:100])
    print(f"\n Perplexité sur le test (100 phrases): {perplexity:.2f}")

    # 5. Sauvegarde
    os.makedirs('models', exist_ok=True)
    trigram_model.save('models/malagasy_trigram.pkl')

    # 6. Démonstration
    demo(trigram_model, test[:10])

    # 7. Modèle Markov (bigrammes) optionnel
    print("\n" + "-"*40)
    print(" Modèle Markov (Bigrammes)")
    print("-"*40)

    markov_model = MarkovChainPredictor()
    markov_model.train(train, vocab_min_freq=2)
    markov_model.save('models/Autocomplétion.pkl')

    print("\n Entraînement terminé !")
    print("   Modèles sauvegardés dans le dossier 'models/'")

if __name__ == "__main__":
    main()

 MODÈLE D'AUTOCOMPLÉTION POUR LE MALGACHE
   N-grammes avec lissage de Laplace
 Chargement du dataset...
 163949 phrases chargées

 Exemples de phrases:
   1. Tenany ihany, fa misy ihany koa ny fampi- fandraisana azy amin- javatra hafa, tahaka ny mikasika ny ...
   2. Teny famaranana eto am-pamaranana dia tsiahivina fa literatiora am-bava miompana amin’ny anatra sy n...
   3. Mauriac, op. Cit. , p 86 aucun drame ne peut commencer de vivre dans mon esprit si je ne le situe da...
   4. Voajanahary ao amin’ny olombelona ny fitiavana ary maro ny endrika isehoany. Misy anefa ny lalàna mi...
   5. Ii misy kosa fitia mandrava ny efa ka mivaona toa toraka diso fiantefa izay voany, marary ka velon-t...

 Division des données:
   Train: 147554 phrases
   Test: 16395 phrases

----------------------------------------
 Modèle Trigrammes (3-grammes)
----------------------------------------
 Entraînement du modèle 3-grammes...
 Vocabulaire: 86092 mots uniques
 8113963 N-grammes enregistrés
 1445580 c

KeyboardInterrupt: Interrupted by user